# ZDT1 HV Comparison

This notebook compares pre-trained DeepIC-assisted EA from `demo.py` against `NSGA-NDA` from `nsga-nda.py` on ZDT1 and plots both hypervolume histories on the same figure.

In [5]:
from pathlib import Path
from types import SimpleNamespace
import importlib.util

import matplotlib.pyplot as plt
import numpy as np

import demo

AttributeError: module 'matplotlib' has no attribute '__version_info__'

In [ ]:
spec = importlib.util.spec_from_file_location("nsga_nda_module", Path("nsga-nda.py"))
nsga_nda = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nsga_nda)

In [ ]:
args = SimpleNamespace(
    dim=30,
    archive_size=80,
    offspring_size=24,
    k_eval=5,
    iterations=8,
    mutation_sigma=0.12,
    kan_steps=25,
    kan_hidden=10,
    kan_grid=5,
    deepic_hidden=64,
    deepic_heads=4,
    deepic_ff=128,
    deepic_lr=1e-3,
    deepic_adapt_steps=8,
    max_fe=160,
    seed=7,
    device="cpu",
)

model_path = Path("deepic_zdt1.pth")
if model_path.exists():
    print(f"Using saved model: {model_path}")
    deepic_model = None
else:
    print("deepic_zdt1.pth not found. Training DeepIC on ZDT1 first...")
    deepic_model = demo.train_deepic_zdt1(args)

In [ ]:
deepic_result = demo.run_infer_zdt1(args, deepic=deepic_model, plot=False)
nsga_result = nsga_nda.run_nsga_nda(args, plot=False)

print("DeepIC final HV:", deepic_result["hv_history"][-1])
print("NSGA-NDA final HV:", nsga_result["hv_history"][-1])
print("Reference point:", deepic_result["ref_point"])
print("DeepIC front size:", len(deepic_result["final_front"]))
print("NSGA-NDA front size:", len(nsga_result["final_front"]))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(deepic_result["hv_history"], marker="o", label="Pre-trained DeepIC-assisted EA")
plt.plot(nsga_result["hv_history"], marker="s", label="NSGA-NDA")
plt.title("ZDT1 Hypervolume Comparison")
plt.xlabel("Step")
plt.ylabel("Hypervolume")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(deepic_result["final_front"][:, 0], deepic_result["final_front"][:, 1], s=22, alpha=0.8, label="DeepIC-assisted EA")
plt.scatter(nsga_result["final_front"][:, 0], nsga_result["final_front"][:, 1], s=22, alpha=0.8, label="NSGA-NDA")
plt.plot(deepic_result["true_front"][:, 0], deepic_result["true_front"][:, 1], "k-", linewidth=2, label="True Pareto Front")
plt.title("ZDT1 Pareto Front Comparison")
plt.xlabel("f1")
plt.ylabel("f2")
plt.grid(True)
plt.legend()
plt.show()